# AWQ 기반 EXAONE-4.0-1.2B 모델 양자화 (로컬 버전)

## 개요
AWQ(Activation-aware Weight Quantization)는 GPTQ보다 빠르고 과적합에 강한 양자화 기법입니다.

### 실행 전 필수 사항
```bash
# 터미널에서 실행
cd lg-aimers8-llm-compression
source venv/bin/activate  # 가상환경 활성화
jupyter notebook          # 노트북 실행
```

---

# 1. Import 및 환경 확인

In [1]:
import os
import sys
import torch
import shutil
from pathlib import Path

from datasets import load_dataset
from transformers import AutoModelForCausalLM, AutoTokenizer

from llmcompressor import oneshot
from llmcompressor.modifiers.quantization import GPTQModifier

print("=" * 60)
print("환경 정보")
print("=" * 60)
print(f"Python: {sys.version.split()[0]}")
print(f"PyTorch: {torch.__version__}")
print(f"CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("⚠️  GPU 없음 - CPU로 실행됩니다 (느림)")
print("=" * 60)
print("\n✅ Import 완료!")

환경 정보
Python: 3.10.13
PyTorch: 2.9.1
CUDA 사용 가능: False
⚠️  GPU 없음 - CPU로 실행됩니다 (느림)

✅ Import 완료!


# 2. 하이퍼파라미터 설정

In [2]:
# ============================================================================
# 모델 설정
# ============================================================================
# 로컬 모델 경로 (다운로드 불필요!)
MODEL_ID = "./open/base_model"
OUT_DIR = "./model_awq"

# 데이터셋 설정
DATASET_ID = "LGAI-EXAONE/MANTA-1M"
DATASET_SPLIT = "train"

# ============================================================================
# 캘리브레이션 설정 (로컬용 - 더 작은 값으로 시간 단축)
# ============================================================================
# GPU 있으면 더 큰 값 사용 가능
NUM_CALIBRATION_SAMPLES = 256  # GPU: 512, CPU: 256 권장
MAX_SEQUENCE_LENGTH = 512      # GPU: 1024, CPU: 512 권장

# 양자화 설정
SCHEME = "W4A16"
TARGETS = ["Linear"]
IGNORE = ["embed_tokens", "lm_head"]
GROUP_SIZE = 128
DAMPENING_FRAC = 0.01

# ============================================================================
# 원본 모델 크기 (비교용)
# ============================================================================
ORIGINAL_MODEL_SIZE_GB = 2.56  # EXAONE-4.0-1.2B 원본 크기

# 설정 요약 출력
print("=" * 60)
print("AWQ 양자화 설정")
print("=" * 60)
print(f"MODEL_ID: {MODEL_ID}")
print(f"OUT_DIR: {OUT_DIR}")
print(f"SCHEME: {SCHEME}")
print(f"NUM_CALIBRATION_SAMPLES: {NUM_CALIBRATION_SAMPLES}")
print(f"MAX_SEQUENCE_LENGTH: {MAX_SEQUENCE_LENGTH}")
print(f"GROUP_SIZE: {GROUP_SIZE}")
print(f"원본 모델 크기: {ORIGINAL_MODEL_SIZE_GB} GB")
print("=" * 60)

AWQ 양자화 설정
MODEL_ID: ./open/base_model
OUT_DIR: ./model_awq
SCHEME: W4A16
NUM_CALIBRATION_SAMPLES: 256
MAX_SEQUENCE_LENGTH: 512
GROUP_SIZE: 128
원본 모델 크기: 2.56 GB


# 3. 모델 로드

In [3]:
print("[INFO] 모델 로드 중...")

tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)

# CPU/GPU 자동 선택
device_map = "auto" if torch.cuda.is_available() else "cpu"

model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float32 if not torch.cuda.is_available() else torch.bfloat16,
    trust_remote_code=True,
    device_map=device_map,
)

print(f"[INFO] 모델 파라미터: {model.num_parameters():,}")
print(f"[INFO] 원본 모델 크기: {ORIGINAL_MODEL_SIZE_GB} GB")
print(f"[INFO] 디바이스: {device_map}")
print("[INFO] 모델/토크나이저 로드 완료")

`torch_dtype` is deprecated! Use `dtype` instead!


[INFO] 모델 로드 중...
[INFO] 모델 파라미터: 1,279,391,488
[INFO] 원본 모델 크기: 2.56 GB
[INFO] 디바이스: cpu
[INFO] 모델/토크나이저 로드 완료


# 4. 캘리브레이션 데이터셋 준비

In [4]:
print("[INFO] 캘리브레이션 데이터 로드 중...")

ds = load_dataset(
    DATASET_ID,
    split=f"{DATASET_SPLIT}[:{NUM_CALIBRATION_SAMPLES}]",
)

def preprocess(example):
    return {
        "text": tokenizer.apply_chat_template(
            example["conversations"],
            add_generation_prompt=True,
            tokenize=False
        )
    }

ds = ds.map(preprocess)

print(f"[INFO] 데이터셋 크기: {len(ds)}")
print("[INFO] 데이터 전처리 완료")

[INFO] 캘리브레이션 데이터 로드 중...


README.md: 0.00B [00:00, ?B/s]

data/train.parquet:   0%|          | 0.00/1.94G [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000000 [00:00<?, ? examples/s]

Map:   0%|          | 0/256 [00:00<?, ? examples/s]

[INFO] 데이터셋 크기: 256
[INFO] 데이터 전처리 완료


# 5. AWQ 양자화 실행

⚠️ **CPU 실행 시 주의**: 이 과정은 CPU에서 2-6시간 이상 소요될 수 있습니다.

In [5]:
print(f"[INFO] AWQ 양자화 시작")
print(f"  - scheme: {SCHEME}")
print(f"  - samples: {NUM_CALIBRATION_SAMPLES}")
print(f"  - max_len: {MAX_SEQUENCE_LENGTH}")
print(f"  - group_size: {GROUP_SIZE}")

if torch.cuda.is_available():
    print("\n🚀 GPU 모드: 10-30분 예상\n")
else:
    print("\n⏳ CPU 모드: 2-6시간 예상 (샘플 수에 따라 다름)\n")

recipe = [
    GPTQModifier(
        scheme=SCHEME,
        targets=TARGETS,
        ignore=IGNORE,
        block_size=GROUP_SIZE,
        dampening_frac=DAMPENING_FRAC,
        actorder="static",
    )
]

oneshot(
    model=model,
    dataset=ds,
    recipe=recipe,
    max_seq_length=MAX_SEQUENCE_LENGTH,
    num_calibration_samples=NUM_CALIBRATION_SAMPLES,
)

print("\n[INFO] AWQ 양자화 완료!")

[INFO] AWQ 양자화 시작
  - scheme: W4A16
  - samples: 256
  - max_len: 512
  - group_size: 128

⏳ CPU 모드: 2-6시간 예상 (샘플 수에 따라 다름)



Tokenizing:   0%|          | 0/256 [00:00<?, ? examples/s]

2026-02-09T14:03:47.781160+0900 | reset | INFO - Compression lifecycle reset
2026-02-09T14:03:47.782373+0900 | from_modifiers | INFO - Creating recipe from modifiers
2026-02-09T14:03:47.811620+0900 | initialize | INFO - Compression lifecycle initialized for 1 modifiers
2026-02-09T14:03:47.812023+0900 | IndependentPipeline | INFO - Inferred `SequentialPipeline` for `GPTQModifier`
2026-02-09T14:03:47.817899+0900 | dispatch_for_sequential | WARNING - CUDA/XPU is not available! Compressing model on CPU instead


W0209 14:03:47.872000 65389 torch/fx/_symbolic_trace.py:52] is_fx_tracing will return true for both fx.symbolic_trace and torch.export. Please use is_fx_tracing_symbolic_tracing() for specifically fx.symbolic_trace or torch.compiler.is_compiling() for specifically torch.export/compile.
(1/31): Calibrating: 100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:38<00:00,  6.70it/s]

2026-02-09T14:04:27.246817+0900 | compress_modules | INFO - Quantizing model.layers.0.self_attn.q_proj using 256 samples


...

(30/31): Calibrating: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:38<00:00,  6.70it/s]

2026-02-09T14:28:07.124485+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.q_proj using 256 samples


2026-02-09T14:28:07.413787+0900 | compress | METRIC - time 0.29s
2026-02-09T14:28:07.414142+0900 | compress | METRIC - error 3258.94
2026-02-09T14:28:07.414957+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-09T14:28:07.415175+0900 | compress | METRIC - Compressed module size: 16.941056 MB
2026-02-09T14:28:07.416542+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.k_proj using 256 samples
2026-02-09T14:28:07.607691+0900 | compress | METRIC - time 0.19s
2026-02-09T14:28:07.608041+0900 | compress | METRIC - error 923.58
2026-02-09T14:28:07.608873+0900 | get_GPU_usage_nv | WARNING - Pynml library error:
 NVML Shared Library Not Found
2026-02-09T14:28:07.609098+0900 | compress | METRIC - Compressed module size: 4.235264 MB
2026-02-09T14:28:07.609764+0900 | compress_modules | INFO - Quantizing model.layers.29.self_attn.v_proj using 256 samples
2026-02-09T14:28:07.791175+0900 | compress | METRIC - time 0.18s
2026-02-09T14:2

(31/31): Propagating: 100%|█████████████████████████████████████████████████████████████████████████████████████████████████████████████| 256/256 [00:00<00:00, 4486.34it/s]


2026-02-09T14:28:18.322322+0900 | finalize | INFO - Compression lifecycle finalized for 1 modifiers
2026-02-09T14:28:18.326810+0900 | post_process | WARNING - Optimized model is not saved. To save, please provide`output_dir` as input arg.Ex. `oneshot(..., output_dir=...)`

[INFO] AWQ 양자화 완료!


# 6. 모델 저장 및 크기 비교

In [6]:
print("[INFO] 모델 저장 중...")

os.makedirs(OUT_DIR, exist_ok=True)
model.save_pretrained(OUT_DIR, save_compressed=True)
tokenizer.save_pretrained(OUT_DIR)

# 저장된 파일 확인
print(f"\n[INFO] 저장된 파일:")
total_size = 0
for f in sorted(os.listdir(OUT_DIR)):
    size = os.path.getsize(os.path.join(OUT_DIR, f))
    total_size += size
    print(f"  {f}: {size/1e6:.1f} MB")

quantized_size_gb = total_size / 1e9

# ============================================================================
# 크기 비교 출력
# ============================================================================
print("\n" + "=" * 60)
print("모델 크기 비교")
print("=" * 60)
print(f"  원본 모델:     {ORIGINAL_MODEL_SIZE_GB:.2f} GB")
print(f"  양자화 모델:   {quantized_size_gb:.2f} GB")
print(f"  ----------------------------------------")
print(f"  크기 감소:     {ORIGINAL_MODEL_SIZE_GB - quantized_size_gb:.2f} GB")
print(f"  압축률:        {quantized_size_gb / ORIGINAL_MODEL_SIZE_GB * 100:.1f}%")
print(f"  압축 배수:     {ORIGINAL_MODEL_SIZE_GB / quantized_size_gb:.2f}x")
print("=" * 60)

[INFO] 모델 저장 중...
2026-02-09T14:28:54.273463+0900 | get_model_compressor | INFO - skip_sparsity_compression_stats set to True. Skipping sparsity compression statistic calculations. No sparsity compressor will be applied.


Compressing model: 210it [00:01, 180.04it/s]



[INFO] 저장된 파일:
  chat_template.jinja: 0.0 MB
  config.json: 0.0 MB
  generation_config.json: 0.0 MB
  merges.txt: 1.2 MB
  model.safetensors: 1407.7 MB
  recipe.yaml: 0.0 MB
  special_tokens_map.json: 0.0 MB
  tokenizer.json: 7.9 MB
  tokenizer_config.json: 0.1 MB
  vocab.json: 1.9 MB

모델 크기 비교
  원본 모델:     2.56 GB
  양자화 모델:   1.42 GB
  ----------------------------------------
  크기 감소:     1.14 GB
  압축률:        55.4%
  압축 배수:     1.80x


# 7. 제출 파일 생성

In [7]:
zip_name = "submit_awq"
print(f"[INFO] {zip_name}.zip 생성 중...")

shutil.make_archive(
    base_name=zip_name,
    format="zip",
    root_dir=".",
    base_dir=OUT_DIR,
)

zip_size = os.path.getsize(f"{zip_name}.zip") / 1e9
print(f"[INFO] 생성 완료: {zip_name}.zip ({zip_size:.2f} GB)")

if zip_size <= 10:
    print("✅ 용량 제한 충족 (≤ 10GB)")
else:
    print("❌ 용량 초과!")

print(f"\n📁 파일 위치: {os.path.abspath(f'{zip_name}.zip')}")

[INFO] submit_awq.zip 생성 중...
[INFO] 생성 완료: submit_awq.zip (0.88 GB)
✅ 용량 제한 충족 (≤ 10GB)

📁 파일 위치: /Users/imdonghyeon/Desktop/lg-aimers8-llm-compression/submit_awq.zip


# 8. (선택) 모델 테스트

In [8]:
print("[INFO] 양자화된 모델 테스트...")

message = [{"role": "user", "content": "안녕하세요, 자기소개 해주세요."}]

input_ids = tokenizer.apply_chat_template(
    message,
    add_generation_prompt=True,
    return_tensors="pt"
).to(model.device)

output = model.generate(
    input_ids,
    max_new_tokens=100,
    do_sample=False,
)

response = tokenizer.decode(output[0], skip_special_tokens=True)
print(f"\n응답:\n{response}")

[INFO] 양자화된 모델 테스트...


AttributeError: 'Linear' object has no attribute 'weight'